# Phase 0 probes

Five questions that are cheap to answer now and expensive to discover by accident
later. None is a blocker. Probe 3 gates the bundle-delivery decision, so do not
leave it until you need the answer.

Run `reference.ipynb` first. If that does not work, nothing here will mean anything.

Record the answers in this notebook as you go.

## Setup — a helper for the rest of the notebook

Each probe gets its own comm, so one failing probe cannot affect another.

In [ ]:
import $ivy.`com.lihaoyi::ujson:4.3.0`
import almond.interpreter.api.DisplayData
import java.nio.charset.StandardCharsets.UTF_8
import java.util.UUID
import scala.collection.mutable.ArrayBuffer

/** Opens an anywidget comm, displays its view, returns the comm id.
  * `onMsg` receives (commId, rawBytes) — the id is handed to the handler, which
  * is how a handler can reply without a forward reference to its own widget. */
def anywidget(
    esm: String,
    state: Seq[(String, ujson.Value)] = Nil
)(onMsg: (String, Array[Byte]) => Unit): String = {
  val id = UUID.randomUUID().toString
  val st = ujson.Obj(
    "_model_module"         -> "anywidget",
    "_model_module_version" -> "~0.11.*",
    "_model_name"           -> "AnyModel",
    "_view_module"          -> "anywidget",
    "_view_module_version"  -> "~0.11.*",
    "_view_name"            -> "AnyView",
    "_esm"                  -> esm
  )
  state.foreach { case (k, v) => st(k) = v }

  val _ = commHandler.sender(
    targetName = "jupyter.widget",
    id         = id,
    data       = ujson.write(ujson.Obj("state" -> st, "buffer_paths" -> ujson.Arr())).getBytes(UTF_8),
    metadata   = ujson.write(ujson.Obj("version" -> "2.0.0")).getBytes(UTF_8),
    onMessage  = onMsg
  )
  publish.display(DisplayData(Map(
    "application/vnd.jupyter.widget-view+json" -> ujson.write(
      ujson.Obj("model_id" -> id, "version_major" -> 2, "version_minor" -> 0))
  )))
  id
}

/** A button that bumps `clicks` and syncs it. Enough to trigger an inbound message. */
def clicker(label: String): String = s"""
function render({ model, el }) {
  const b = document.createElement("button");
  b.innerHTML = "$label";
  b.addEventListener("click", () => {
    model.set("clicks", (model.get("clicks") || 0) + 1);
    model.save_changes();
  });
  el.appendChild(b);
}
export default { render };
"""

## Probe 1 — where does handler output go?

The handler runs on a kernel thread, outside any cell execution, so there is no
obvious parent message to attach output to.

This decides how errors and progress get reported from a callback, which matters
as soon as a control triggers a real calculation. It is the problem ipywidgets
solved with its `Output` widget.

**Click the button below, then look in four places:** under this cell; under
whichever cell is currently running; the Jupyter output channel (View → Output →
"Jupyter"); nowhere at all.

In [ ]:
val p1 = anywidget(clicker("probe 1: click me"), Seq("clicks" -> ujson.Num(0))) { (_, bytes) =>
  println(s"[probe1] println from handler: ${new String(bytes, UTF_8)}")
  Console.err.println("[probe1] Console.err from handler")
}

**Answer (fill in):**

- `println` landed: 
- `Console.err` landed: 

## Probe 2 — what happens to an exception in a handler?

The question that matters is not whether the exception is visible. It is whether
the **comm survives it**. If an escaping exception silently kills the comm, every
handler needs a `Try` wrapper from day one.

Click the button **three times**, then run the cell after it.

In [ ]:
val p2seen = ArrayBuffer.empty[String]

val p2 = anywidget(clicker("probe 2: click me 3 times"), Seq("clicks" -> ujson.Num(0))) { (_, bytes) =>
  p2seen.synchronized { p2seen += new String(bytes, UTF_8) }
  throw new RuntimeException("[probe2] deliberate handler failure")
}

In [ ]:
// 3 => the comm survives an escaping exception.
// 1 => the first throw killed it; wrapping is mandatory, not merely prudent.
p2seen.synchronized(p2seen.size)

**Answer (fill in):** messages received after the first throw = 

Note: `wedgie.kernel.Widget` wraps every handler in a `try` regardless of what this
says. The answer only changes how loudly the failure needs to be surfaced.

## Probe 3 — can the webview import a module from an internal URL?

**This is the one that decides the architecture.**

If it passes: serve the Scala.js bundle, `_esm` stays at ~120 bytes, and the
`.ipynb` stays small. If it fails: the bundle must either be inlined (~324 KB per
widget, in the notebook file, in git) or delivered over the comm (probes 4 and 5).

Two complications specific to this setup:

- `_esm` is evaluated from a **blob URL**, so relative imports cannot resolve.
  Every import must be absolute.
- The webview runs on the **Windows** side while the kernel is in WSL2 (visible as
  `remoteAuthority=wsl+ubuntu-hardened` in the webview URL). A bundle served from
  inside WSL2 on `localhost` will not be reachable. Bind it somewhere Windows can
  reach and use the real hostname.

The jsdelivr import is the **control**: it distinguishes a CSP refusal (both fail)
from a networking or MIME-type problem at your host (only yours fails).

Serve a file at your internal URL containing exactly:

```js
export const ok = "internal module loaded";
```

served with `Content-Type: text/javascript`.

In [ ]:
// Must be absolute, and reachable from the Windows-side webview.
val internalUrl = "https://REPLACE-ME.internal/wedgie-probe.js"
val controlUrl  = "https://cdn.jsdelivr.net/npm/anywidget@0.11/+esm"

val p3 = anywidget(s"""
function line(el, text) {
  const p = document.createElement("pre");
  p.style.margin = "2px 0";
  p.textContent = text;
  el.appendChild(p);
  return p;
}

function render({ model, el }) {
  const control  = line(el, "control  (jsdelivr): ...");
  const internal = line(el, "internal (yours)   : ...");

  import("$controlUrl")
    .then(m  => control.textContent  = "control  (jsdelivr): PASS — " + Object.keys(m).slice(0, 3).join(", "))
    .catch(e => control.textContent  = "control  (jsdelivr): FAIL — " + e);

  import("$internalUrl")
    .then(m  => internal.textContent = "internal (yours)   : PASS — " + JSON.stringify(m.ok))
    .catch(e => internal.textContent = "internal (yours)   : FAIL — " + e);
}
export default { render };
""") { (_, _) => () }

**Reading the result:**

| control | internal | means |
| --- | --- | --- |
| PASS | PASS | Serve the bundle. `_esm` stays tiny. |
| PASS | FAIL | Not CSP — your host is unreachable or serving the wrong MIME type. Check the WSL2/Windows split first. |
| FAIL | FAIL | Dynamic `import()` from `_esm` is blocked outright. Fall back to inline or comm delivery. |

**Answer (fill in):** control = , internal = 

## Probe 4 — do `custom` messages round-trip?

Needed by the comm-delivered bundle strategy, and useful as a general diagnostic:
if `custom` arrives but `update` does nothing, the problem is update semantics; if
neither arrives, it is delivery.

Click the button. The `<pre>` should fill in with the kernel's reply.

In [ ]:
val p4seen = ArrayBuffer.empty[String]

val p4 = anywidget("""
function render({ model, el }) {
  const out = document.createElement("pre");
  out.textContent = "probe 4: no reply yet";
  el.appendChild(out);

  model.on("msg:custom", (msg) => {
    out.textContent = "probe 4: kernel replied -> " + JSON.stringify(msg);
  });

  const b = document.createElement("button");
  b.innerHTML = "probe 4: send custom to kernel";
  b.addEventListener("click", () => model.send({ hello: "from frontend" }));
  el.appendChild(b);
}
export default { render };
""") { (id, bytes) =>
  p4seen.synchronized { p4seen += new String(bytes, UTF_8) }
  // The handler is handed its own comm id, so it can reply without a forward reference.
  commHandler.commMessage(
    id,
    ujson.write(ujson.Obj(
      "method"  -> "custom",
      "content" -> ujson.Obj("hello" -> "from kernel")
    )).getBytes(UTF_8),
    "{}".getBytes(UTF_8)
  )
}

In [ ]:
// Should contain {"method":"custom","content":{"hello":"from frontend"}}
p4seen.synchronized(p4seen.toList).foreach(println)

**Answer (fill in):** kernel → frontend = , frontend → kernel = 

## Probe 5 — does anywidget accept an async `render`?

The comm-delivered strategy needs `render` to be able to await the bundle before
mounting. If this renders the success line, that shape is available.

In [ ]:
val p5 = anywidget("""
export default {
  async render({ model, el }) {
    el.textContent = "probe 5: render started, awaiting...";
    await new Promise((r) => setTimeout(r, 400));
    const p = document.createElement("pre");
    p.textContent = "probe 5: PASS — async render resolved and mounted";
    el.replaceChildren(p);
  }
};
""") { (_, _) => () }

**Answer (fill in):** async render = 

---

## Decision

Probes 4 and 5 both passing makes `EsmSource.CommDelivered` viable, which keeps the
bundle out of the `.ipynb` **without** needing probe 3. That is the outcome that
removes the one-widget-per-notebook constraint entirely.

Fill in and commit:

| Probe | Answer | Consequence |
| --- | --- | --- |
| 1 handler output | | where failures get reported |
| 2 handler exception | | how loudly to surface them |
| 3 internal import | | `RemoteImport` viable? |
| 4 custom round-trip | | `CommDelivered` viable? |
| 5 async render | | `CommDelivered` viable? |